In [ ]:
import pandas as pd
from collections import defaultdict

FILE_PATH = "C:/Users/Ex0164/Tushar vats/Copy of Master Data _290102026.xlsx"
MAIN_SHEET = "Master Data "
PPM_SHEET = "Part Production Master "

ALLOWED_MACHINES = ["MP-01", "MP-05", "MP-10", "MP-17"]

def main():
    xls = pd.ExcelFile(FILE_PATH)
    
    # Load main data
    df = pd.read_excel(xls, MAIN_SHEET)
    df.columns = df.columns.str.strip()
    
    # Optional: load cycle time override from PPM (Machine column = cycle time)
    try:
        df_cycle = pd.read_excel(xls, PPM_SHEET)
        df_cycle.columns = df_cycle.columns.str.strip()
        cycle_map = dict(zip(
            df_cycle["Material"].astype(str).str.strip(),
            pd.to_numeric(df_cycle["Machine"], errors='coerce')
        ))
        df["Cycle Time"] = df["Child Part"].astype(str).str.strip().map(cycle_map).fillna(df["Cycle Time"])
    except:
        print("Note: Could not load cycle time from Part Production Master. Using main sheet values.")
    
    # Filter rows with positive Daily Plan
    df = df[df["Daily Plan"].fillna(0) > 0].copy()
    
    # Quantity to produce
    df["qty"] = (df["Plan"] - df["Inventory_25"].fillna(0)).clip(lower=0)
    
    # Only rows needing production
    df_need = df[df["qty"] > 0].copy()
    
    if df_need.empty:
        print("\nNo parts need production (all covered by Inventory_25).")
        return
    
    # Load tracking (in hours)
    machine_load = {m: 0.0 for m in ALLOWED_MACHINES}
    plan_rows = []
    
    for _, row in df_need.iterrows():
        child = str(row["Child Part"]).strip()
        vm_str = str(row.get("Vertical Machines", ""))
        possible_machines = [m.strip() for m in vm_str.split(",") if m.strip() in ALLOWED_MACHINES]
        
        if not possible_machines:
            continue
        
        # Choose machine with least current load
        machine = min(possible_machines, key=lambda m: machine_load[m])
        
        # Time in hours (Cycle Time assumed in seconds)
        time_hours = (row["qty"] * row["Cycle Time"]) / 3600.0
        
        machine_load[machine] += time_hours
        
        plan_rows.append({
            "Child Part": child,
            "Switch Part": row["Switch Part Number"],
            "Machine": machine,
            "Quantity": int(round(row["qty"])),
            "Time (hours)": round(time_hours, 2)
        })
    
    if not plan_rows:
        print("\nNo valid production assignments (no eligible machines found in Vertical Machines).")
        return
    
    df_plan = pd.DataFrame(plan_rows)
    
    # Sort plan by machine and then by time descending
    df_plan = df_plan.sort_values(["Machine", "Time (hours)"], ascending=[True, False])
    
    # Load summary
    df_load = pd.DataFrame({
        "Machine": list(machine_load.keys()),
        "Total Load (hours)": [round(v, 2) for v in machine_load.values()]
    }).sort_values("Machine")
    
    # Display directly
    print("\n" + "="*70)
    print("          PRODUCTION PLAN (per row - independent)")
    print("="*70)
    print(df_plan.to_string(index=False))
    
    print("\n" + "="*70)
    print("          MACHINE LOAD SUMMARY")
    print("="*70)
    print(df_load.to_string(index=False))
    
    print("\nDone. No Excel file created.")


if __name__ == "__main__":
    main()